<a href="https://colab.research.google.com/github/CMadhan21/dvcon2026-task-aware-detection/blob/main/DVCon2026_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ════════════════════════════════════════════════════════════════════════════
# DVCon India 2026 — Task-Aware Object Detection Pipeline
# Member 1: Madhan C (CMadhan21) — Complete Working Code
# CLIP + YOLOv8 + Interactive UI (URL / File Upload / Webcam)
# ════════════════════════════════════════════════════════════════════════════

# ── Installations ─────────────────────────────────────────────────────────────
import subprocess
subprocess.run(["pip", "install",
                "git+https://github.com/openai/CLIP.git",
                "-q"], check=True)
subprocess.run(["pip", "install", "ultralytics",    "-q"], check=True)
subprocess.run(["pip", "install", "ipywidgets",     "-q"], check=True)

# ── Imports ───────────────────────────────────────────────────────────────────
import os, io, torch, clip, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import ipywidgets as widgets
from PIL import Image
from ultralytics import YOLO
from IPython.display import display, HTML, clear_output

os.makedirs("results", exist_ok=True)

# ════════════════════════════════════════════════════════════════════════════
# 1. MODELS (Fixed for Ultralytics + PyTorch 2.6 Custom Modules)
# ════════════════════════════════════════════════════════════════════════════
import torch
from ultralytics import YOLO

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load CLIP
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()

# Load YOLO with weights_only=False inside a safe context
# This bypasses the UnpicklingError for custom Ultralytics layers
with torch.serialization.safe_globals([torch.nn.modules.container.Sequential]):
    try:
        # Some environments allow passing it directly via ultralytics
        yolo_model = YOLO("yolov8n.pt")
    except Exception:
        # If the above fails, we force the global torch setting just for this line
        import torch.serialization
        _original_load = torch.load
        torch.load = lambda *args, **kwargs: _original_load(*args, **{**kwargs, 'weights_only': False})
        yolo_model = YOLO("yolov8n.pt")
        torch.load = _original_load # Restore original function

print(f"✅ CLIP loaded    : ViT-B/32 on {device}")
print(f"✅ YOLO loaded    : YOLOv8-nano")
# ════════════════════════════════════════════════════════════════════════════
# 2. TASK DEFINITIONS
# ════════════════════════════════════════════════════════════════════════════
TASKS = {
    1:  "step on something to reach the top of a shelf",
    2:  "sit comfortably",
    3:  "place flowers in something",
    4:  "get potatoes out of fire",
    5:  "water a plant",
    6:  "get lemon out of tea",
    7:  "dig a hole",
    8:  "open a bottle of beer",
    9:  "open a parcel or box",
    10: "serve wine",
    11: "pour sugar",
    12: "smear butter on bread",
    13: "extinguish a fire",
    14: "pound or beat a carpet",
}

TASK_PROMPTS = {
    1:  ["a stool", "a chair", "a ladder", "a box to stand on", "a step"],
    2:  ["a comfortable sofa", "a couch", "a soft chair",
         "a bed", "a cushioned seat"],
    3:  ["a vase", "a pot", "a jar",
         "a container for flowers", "a flower pot"],
    4:  ["tongs", "a ladle", "a spoon",
         "a fork", "a spatula", "a cooking tool"],
    5:  ["a watering can", "a bottle", "a hose",
         "a cup of water", "a jug"],
    6:  ["a spoon", "a fork", "tongs", "a ladle", "a stirrer"],
    7:  ["a shovel", "a spade", "a trowel",
         "a digging tool", "a fork"],
    8:  ["a bottle opener", "a corkscrew", "a knife", "an opener"],
    9:  ["scissors", "a knife", "a box cutter", "a blade", "a cutter"],
    10: ["a wine glass", "a cup", "a goblet",
         "a glass", "a drinking vessel"],
    11: ["a spoon", "a ladle", "a scoop", "a cup", "a bowl"],
    12: ["a knife", "a butter knife", "a spatula", "a spreader"],
    13: ["a fire extinguisher", "a bucket",
         "a blanket", "water", "a hose"],
    14: ["a carpet beater", "a broom", "a stick", "a brush", "a bat"],
}

# ════════════════════════════════════════════════════════════════════════════
# 3. ENCODE TASK PROMPTS
# ════════════════════════════════════════════════════════════════════════════
def encode_task_prompts(task_prompts):
    task_embeddings = {}
    with torch.no_grad():
        for task_id, prompts in task_prompts.items():
            texts     = [f"a photo of {p}" for p in prompts]
            tokens    = clip.tokenize(texts).to(device)
            embeds    = clip_model.encode_text(tokens)
            embeds    = embeds / embeds.norm(dim=-1, keepdim=True)
            avg_embed = embeds.mean(dim=0, keepdim=True)
            avg_embed = avg_embed / avg_embed.norm(dim=-1, keepdim=True)
            task_embeddings[task_id] = avg_embed
    return task_embeddings

task_embeddings = encode_task_prompts(TASK_PROMPTS)
print(f"✅ Task embeddings: {len(task_embeddings)} tasks encoded")

# ════════════════════════════════════════════════════════════════════════════
# 4. DETECTION MODULE
# ════════════════════════════════════════════════════════════════════════════
def detect_and_crop(image, conf_threshold=0.20):
    results = yolo_model(image, conf=conf_threshold, verbose=False)
    result  = results[0]
    crops, labels, boxes, confidences = [], [], [], []
    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        x1 = max(0, x1);  y1 = max(0, y1)
        x2 = min(image.width, x2);  y2 = min(image.height, y2)
        if (x2 - x1) < 10 or (y2 - y1) < 10:
            continue
        crop     = image.crop((x1, y1, x2, y2))
        class_id = int(box.cls[0].item())
        label    = yolo_model.names[class_id]
        conf     = round(float(box.conf[0].item()), 3)
        crops.append(crop);    labels.append(label)
        boxes.append([x1, y1, x2, y2]);  confidences.append(conf)
    return crops, labels, boxes, confidences

# ════════════════════════════════════════════════════════════════════════════
# 5. CLIP SCORING MODULE
# ════════════════════════════════════════════════════════════════════════════
def encode_image_crop(crop):
    w, h   = crop.size
    maxdim = max(w, h)
    padded = Image.new("RGB", (maxdim, maxdim), (128, 128, 128))
    padded.paste(crop, ((maxdim - w) // 2, (maxdim - h) // 2))
    inp = clip_preprocess(padded).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = clip_model.encode_image(inp)
        emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb

def rank_objects(crops, labels, task_id, confidences=None):
    results = []
    for i, (crop, label) in enumerate(zip(crops, labels)):
        img_emb    = encode_image_crop(crop)
        txt_emb    = task_embeddings[task_id]
        clip_score = (img_emb @ txt_emb.T).item()
        yolo_conf  = confidences[i] if confidences else 1.0
        # CLIP drives 85%, YOLO confidence fine-tunes 15%
        final      = clip_score * (0.85 + 0.15 * yolo_conf)
        results.append({
            "rank":       0,
            "label":      label,
            "clip_score": round(clip_score, 4),
            "yolo_conf":  round(yolo_conf,  3),
            "score":      round(final,      4),
            "index":      i,
        })
    results = sorted(results, key=lambda x: x["score"], reverse=True)
    for i, r in enumerate(results):
        r["rank"] = i + 1
    return results

# ════════════════════════════════════════════════════════════════════════════
# 6. VISUALISATION + PIPELINE
# ════════════════════════════════════════════════════════════════════════════
RANK_COLORS = {1: "#00C853", 2: "#FF6D00", 3: "#D50000"}

def run_and_show(image, task_id, output_widget):
    with output_widget:
        clear_output(wait=True)

        crops, labels, boxes, confidences = detect_and_crop(image)
        if not crops:
            print("⚠ No objects detected. Try a clearer/closer image.")
            return

        ranked = rank_objects(crops, labels, task_id, confidences)
        for r in ranked:
            r["box"] = boxes[r["index"]]

        # ── Figure: annotated image + bar chart ──────────────────────────────
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        # Left — image with bounding boxes
        ax = axes[0]
        ax.imshow(image)
        for r in ranked:
            x1, y1, x2, y2 = r["box"]
            color = RANK_COLORS.get(r["rank"], "#9E9E9E")
            lw    = 3 if r["rank"] <= 3 else 1
            alpha = 1.0 if r["rank"] <= 3 else 0.18
            ax.add_patch(patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=lw, edgecolor=color,
                facecolor="none", alpha=alpha))
            if r["rank"] <= 3:
                ax.text(x1, y1-6,
                        f"#{r['rank']} {r['label']} ({r['score']:.3f})",
                        fontsize=9, color="white", fontweight="bold",
                        bbox=dict(facecolor=color, alpha=0.85,
                                  pad=2, edgecolor="none"))
        ax.set_title(f'Task {task_id}: "{TASKS[task_id]}"',
                     fontsize=11, fontweight="bold", pad=10)
        ax.axis("off")

        # Right — horizontal bar chart top-5
        ax2    = axes[1]
        top5   = ranked[:min(5, len(ranked))]
        names  = [f"#{r['rank']} {r['label']}" for r in top5]
        scores = [r["score"] for r in top5]
        bclrs  = [RANK_COLORS.get(r["rank"], "#78909C") for r in top5]
        bars   = ax2.barh(names[::-1], scores[::-1], color=bclrs[::-1],
                          edgecolor="white", linewidth=0.5)
        for bar, score in zip(bars, scores[::-1]):
            ax2.text(bar.get_width() + 0.0008,
                     bar.get_y() + bar.get_height() / 2,
                     f"{score:.4f}", va="center", fontsize=9,
                     fontweight="bold")
        ax2.set_xlabel("Final Score  (CLIP × YOLO weight)", fontsize=9)
        ax2.set_title("Top-5 Object Rankings", fontsize=11, fontweight="bold")
        ax2.set_xlim(0, max(scores) * 1.18)
        ax2.spines["top"].set_visible(False)
        ax2.spines["right"].set_visible(False)

        plt.suptitle("DVCon India 2026 — Task-Aware Object Detection",
                     fontsize=13, fontweight="bold", y=1.01)
        plt.tight_layout()
        save_path = f"results/task_{task_id}_result.png"
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.show()

        # ── Text ranking table ────────────────────────────────────────────────
        print(f"\n  Task {task_id}: {TASKS[task_id]}")
        print(f"  Objects detected : {len(crops)}")
        print(f"  {'─'*54}")
        print(f"  {'Rank':<6}{'Label':<22}{'CLIP':<10}{'YOLO':<8}{'Final'}")
        print(f"  {'─'*54}")
        for r in ranked:
            mk = "  ◀ BEST PICK" if r["rank"] == 1 else ""
            print(f"  {r['rank']:<6}{r['label']:<22}"
                  f"{r['clip_score']:<10}{r['yolo_conf']:<8}"
                  f"{r['score']}{mk}")
        print(f"\n  ✅ Saved → {save_path}")

# ════════════════════════════════════════════════════════════════════════════
# 7. INTERACTIVE UI
# ════════════════════════════════════════════════════════════════════════════
task_options = [(f"Task {k}: {v}", k) for k, v in TASKS.items()]

main_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #BDBDBD",
        border_radius="8px",
        padding="12px",
        min_height="80px",
    )
)
with main_output:
    print("  Results will appear here after you run the pipeline.")

# ── TAB 1: URL ────────────────────────────────────────────────────────────────
url_task   = widgets.Dropdown(options=task_options, value=10,
                               description="Task:",
                               layout=widgets.Layout(width="500px"),
                               style={"description_width": "50px"})
url_input  = widgets.Text(
                placeholder="Paste direct image URL (jpg / png)...",
                layout=widgets.Layout(width="450px"))
url_btn    = widgets.Button(description="▶ Run",
                             button_style="primary",
                             layout=widgets.Layout(width="80px", height="36px"))
url_status = widgets.Label(value="")

def on_url(b):
    url = url_input.value.strip()
    if not url:
        url_status.value = "⚠ Enter a URL first."; return
    url_status.value = "⏳ Loading image..."
    try:
        import requests
        from io import BytesIO
        r     = requests.get(url, timeout=15,
                             headers={"User-Agent": "Mozilla/5.0"})
        image = Image.open(BytesIO(r.content)).convert("RGB")
        url_status.value = f"✅ Loaded: {image.width}×{image.height}px"
        run_and_show(image, url_task.value, main_output)
    except Exception as e:
        url_status.value = f"✗ Error: {e}"

url_btn.on_click(on_url)

tab1 = widgets.VBox([
    widgets.HTML(
        "<p style='font-family:Arial;font-weight:bold;"
        "color:#1565C0;margin:4px 0'>🔗 Paste Image URL</p>"),
    url_task,
    widgets.HBox([url_input, url_btn]),
    url_status,
], layout=widgets.Layout(padding="10px"))

# ── TAB 2: FILE UPLOAD ────────────────────────────────────────────────────────
file_task   = widgets.Dropdown(options=task_options, value=10,
                                description="Task:",
                                layout=widgets.Layout(width="500px"),
                                style={"description_width": "50px"})
file_upload = widgets.FileUpload(accept="image/*", multiple=False,
                                  layout=widgets.Layout(width="300px"))
file_btn    = widgets.Button(description="▶ Run",
                              button_style="success",
                              layout=widgets.Layout(width="80px", height="36px"))
file_status = widgets.Label(value="")

def on_file(b):
    if not file_upload.value:
        file_status.value = "⚠ Upload an image first."; return
    file_status.value = "⏳ Processing..."
    try:
        uploaded = list(file_upload.value.values())[0]
        content  = uploaded["content"] if isinstance(uploaded, dict) \
                   else bytes(uploaded)
        image    = Image.open(io.BytesIO(content)).convert("RGB")
        file_status.value = f"✅ Loaded: {image.width}×{image.height}px"
        run_and_show(image, file_task.value, main_output)
    except Exception as e:
        file_status.value = f"✗ Error: {e}"

file_btn.on_click(on_file)

tab2 = widgets.VBox([
    widgets.HTML(
        "<p style='font-family:Arial;font-weight:bold;"
        "color:#2E7D32;margin:4px 0'>📁 Upload Image File</p>"),
    file_task,
    widgets.HBox([file_upload, file_btn]),
    file_status,
], layout=widgets.Layout(padding="10px"))

# ── TAB 3: WEBCAM ─────────────────────────────────────────────────────────────
webcam_task   = widgets.Dropdown(options=task_options, value=10,
                                  description="Task:",
                                  layout=widgets.Layout(width="500px"),
                                  style={"description_width": "50px"})
webcam_upload = widgets.FileUpload(accept="image/*", multiple=False,
                                    layout=widgets.Layout(width="300px"))
webcam_btn    = widgets.Button(description="▶ Run Pipeline",
                                button_style="warning",
                                layout=widgets.Layout(width="130px",
                                                      height="36px"))
webcam_status = widgets.Label(value="")

def on_webcam(b):
    if not webcam_upload.value:
        webcam_status.value = "⚠ Upload captured image first."; return
    webcam_status.value = "⏳ Processing..."
    try:
        uploaded = list(webcam_upload.value.values())[0]
        content  = uploaded["content"] if isinstance(uploaded, dict) \
                   else bytes(uploaded)
        image    = Image.open(io.BytesIO(content)).convert("RGB")
        webcam_status.value = f"✅ Loaded: {image.width}×{image.height}px"
        run_and_show(image, webcam_task.value, main_output)
    except Exception as e:
        webcam_status.value = f"✗ Error: {e}"

webcam_btn.on_click(on_webcam)

tab3 = widgets.VBox([
    widgets.HTML("""
    <p style='font-family:Arial;font-weight:bold;
       color:#E65100;margin:4px 0'>📷 Webcam Capture</p>
    <div style='background:#1A1A2E;padding:14px;
                border-radius:8px;font-family:Arial;'>
      <video id="wc_vid" autoplay playsinline
             style='width:460px;height:320px;background:#000;
                    border:2px solid #1565C0;border-radius:6px;
                    display:block;margin-bottom:10px;'></video>
      <canvas id="wc_canvas" style='display:none'></canvas>
      <button onclick='wcStart()'
        style='background:#1565C0;color:white;border:none;
               padding:8px 18px;border-radius:5px;
               cursor:pointer;margin-right:8px;font-size:13px;'>
        📷 Start Camera
      </button>
      <button onclick='wcCapture()' id='wc_cap_btn' disabled
        style='background:#2E7D32;color:white;border:none;
               padding:8px 18px;border-radius:5px;
               cursor:pointer;font-size:13px;opacity:0.5;'>
        📸 Capture &amp; Download
      </button>
      <p id='wc_status'
         style='color:#90CAF9;font-size:12px;margin:8px 0 0 0;'>
        Click Start Camera to begin.
      </p>
    </div>
    <script>
    let wc_stream=null;
    async function wcStart(){
      try{
        wc_stream=await navigator.mediaDevices.getUserMedia({video:true});
        document.getElementById('wc_vid').srcObject=wc_stream;
        await document.getElementById('wc_vid').play();
        const b=document.getElementById('wc_cap_btn');
        b.disabled=false; b.style.opacity='1';
        document.getElementById('wc_status').innerText=
          '✅ Camera live! Click Capture & Download.';
      }catch(e){
        document.getElementById('wc_status').innerText='⚠ '+e.message;
      }
    }
    function wcCapture(){
      const v=document.getElementById('wc_vid');
      const c=document.getElementById('wc_canvas');
      c.width=v.videoWidth; c.height=v.videoHeight;
      c.getContext('2d').drawImage(v,0,0);
      if(wc_stream) wc_stream.getTracks().forEach(t=>t.stop());
      c.toBlob(blob=>{
        const a=document.createElement('a');
        a.href=URL.createObjectURL(blob);
        a.download='webcam_capture.jpg'; a.click();
      },'image/jpeg',0.95);
      document.getElementById('wc_status').innerText=
        '✅ Downloaded! Upload webcam_capture.jpg below then click Run.';
    }
    </script>
    """),
    webcam_task,
    widgets.HTML(
        "<p style='font-family:Arial;font-size:12px;"
        "color:#555;margin:4px 0'>"
        "📁 Upload the downloaded webcam_capture.jpg here:</p>"),
    widgets.HBox([webcam_upload, webcam_btn, webcam_status]),
], layout=widgets.Layout(padding="10px"))

# ── TABS ASSEMBLY ─────────────────────────────────────────────────────────────
tabs = widgets.Tab(children=[tab1, tab2, tab3])
tabs.set_title(0, "🔗 URL")
tabs.set_title(1, "📁 File Upload")
tabs.set_title(2, "📷 Webcam")

# ── FULL UI DISPLAY ───────────────────────────────────────────────────────────
display(HTML("""
<div style="background:linear-gradient(135deg,#0D1B2A,#1565C0);
            padding:16px 22px;border-radius:10px;margin-bottom:12px;">
  <h2 style="color:white;margin:0;font-family:Arial;">
    🎯 DVCon India 2026 — Task-Aware Object Detection
  </h2>
  <p style="color:#90CAF9;margin:5px 0 0 0;
            font-family:Arial;font-size:12px;">
    CLIP ViT-B/32 + YOLOv8-nano &nbsp;|&nbsp;
    14 COCO Tasks &nbsp;|&nbsp;
    GCE Erode — Madhan C
  </p>
</div>
"""))

display(widgets.VBox([
    tabs,
    widgets.HTML("<hr style='margin:10px 0;border-color:#E0E0E0'>"),
    widgets.HTML(
        "<b style='font-family:Arial;color:#1565C0;'>"
        "📊 Pipeline Output</b>"),
    main_output,
]))

# ════════════════════════════════════════════════════════════════════════════
# 8. SAVE TO GITHUB
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*55)
print("  DAY 1 COMPLETE ✅ — Saving to GitHub...")
print("═"*55)

import subprocess

# Clone repo
os.chdir("/content")
if not os.path.exists("dvcon2026-task-aware-detection"):
    subprocess.run([
        "git", "clone",
        "https://github.com/CMadhan21/dvcon2026-task-aware-detection.git"
    ], check=True)

os.chdir("/content/dvcon2026-task-aware-detection")

# Create src folder and save pipeline as .py
os.makedirs("src", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("data", exist_ok=True)

# Copy notebook
subprocess.run([
    "cp", "/content/DVCon2026_Pipeline.ipynb",
    "/content/dvcon2026-task-aware-detection/"
], check=False)

# Write README
readme = """# DVCon India 2026 — Task-Aware Object Detection

**Team: GCE Erode**
Madhan C · Sriramprasath A · Michael Raj A · Mohammad Thoufiq

## Pipeline
- **Detection**: YOLOv8-nano (INT8 quantized for FPGA)
- **Scoring**: CLIP ViT-B/32 semantic similarity
- **Re-ranking**: CLIP score × YOLO confidence weighting
- **Deployment target**: Genesys-2 FPGA + VEGA Processor (CDAC)

## Dataset
- COCO-Tasks: ~40,000 images, 14 tasks
- Reference: Sawatzky et al. (2019)

## Setup
```bash
pip install git+https://github.com/openai/CLIP.git
pip install ultralytics torch torchvision
```

## Run
Open `DVCon2026_Pipeline.ipynb` in Google Colab.
Select task → input image (URL / file / webcam) → view results.

## Structure
```
src/          — pipeline source code
data/         — test images
results/      — output images with bounding boxes
models/       — quantized model weights
```
"""
with open("README.md", "w") as f:
    f.write(readme)

# Git config and push
subprocess.run(["git", "config", "user.email",
                "madhan@gceerode.ac.in"], check=False)
subprocess.run(["git", "config", "user.name",
                "CMadhan21"], check=False)
subprocess.run(["git", "add", "."], check=True)
subprocess.run(["git", "commit", "-m",
                "Day 1 complete: CLIP+YOLOv8 pipeline + interactive UI"],
               check=False)
subprocess.run(["git", "push", "origin", "main"], check=False)

print("✅ GitHub updated!")
print(f"   → github.com/CMadhan21/dvcon2026-task-aware-detection")

# ════════════════════════════════════════════════════════════════════════════
# 9. TEAM STATUS SUMMARY
# ════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*58)
print("  DVCon India 2026 — DAY 1 STATUS")
print("═"*58)

status = [
    ("M1", "Madhan C",          "Pipeline Architect",      [
        ("GitHub repo setup",            "✅ Done"),
        ("CLIP scoring module",          "✅ Done"),
        ("YOLOv8 detection module",      "✅ Done"),
        ("Full pipeline integration",    "✅ Done"),
        ("Interactive UI (3 inputs)",    "✅ Done"),
        ("Batch mAP@0.5 evaluation",     "⏳ Day 2"),
    ]),
    ("M2", "Sriramprasath A",   "Detection & Dataset",     [
        ("YOLOv8-nano setup",            "⏳ Pending"),
        ("COCO-Tasks annotation loader", "⏳ Pending"),
        ("Evaluation dataset prep",      "⏳ Pending"),
    ]),
    ("M3", "Michael Raj A",     "FPGA Architecture",       [
        ("Genesys-2 / VEGA study",       "⏳ Pending"),
        ("Hardware block diagram",       "⏳ Pending"),
        ("Verilog dot-product unit",     "⏳ Pending"),
    ]),
    ("M4", "Mohammad Thoufiq",  "Proposal Writer",         [
        ("DVCon template setup",         "⏳ Pending"),
        ("Section 1 & 2 draft",          "⏳ Pending"),
        ("Pipeline diagram figure",      "⏳ Pending"),
    ]),
]

for mid, name, role, tasks in status:
    print(f"\n  {mid} — {name} ({role})")
    print(f"  {'─'*46}")
    for task, stat in tasks:
        print(f"    {stat}  {task}")

print("\n" + "═"*58)
print("  DAY 2 TARGETS")
print("═"*58)
day2 = [
    ("M1 Madhan",       "Batch mAP@0.5 eval (all 14 tasks) + GitHub cleanup"),
    ("M2 Sriramprasath","INT8 quantization + latency benchmarking"),
    ("M3 Michael Raj",  "Finalize Verilog + resource/power estimates"),
    ("M4 Thoufiq",      "Assemble final proposal + submit"),
]
for who, task in day2:
    print(f"  ▶ {who:<18} {task}")

print("═"*58)
print("\n  🔗 GitHub: github.com/CMadhan21/dvcon2026-task-aware-detection")
print("  📅 Next: Day 2 — Batch evaluation + proposal submission")
print("  🎯 Target: Beat 32.6% mAP baseline from Sawatzky et al.\n")

✅ CLIP loaded    : ViT-B/32 on cuda
✅ YOLO loaded    : YOLOv8-nano
✅ Task embeddings: 14 tasks encoded



═══════════════════════════════════════════════════════
  DAY 1 COMPLETE ✅ — Saving to GitHub...
═══════════════════════════════════════════════════════
✅ GitHub updated!
   → github.com/CMadhan21/dvcon2026-task-aware-detection

══════════════════════════════════════════════════════════
  DVCon India 2026 — DAY 1 STATUS
══════════════════════════════════════════════════════════

  M1 — Madhan C (Pipeline Architect)
  ──────────────────────────────────────────────
    ✅ Done  GitHub repo setup
    ✅ Done  CLIP scoring module
    ✅ Done  YOLOv8 detection module
    ✅ Done  Full pipeline integration
    ✅ Done  Interactive UI (3 inputs)
    ⏳ Day 2  Batch mAP@0.5 evaluation

  M2 — Sriramprasath A (Detection & Dataset)
  ──────────────────────────────────────────────
    ⏳ Pending  YOLOv8-nano setup
    ⏳ Pending  COCO-Tasks annotation loader
    ⏳ Pending  Evaluation dataset prep

  M3 — Michael Raj A (FPGA Architecture)
  ──────────────────────────────────────────────
    ⏳ Pending  Ge